# M1 Chat SFT Data Generation (`m1_teacher_gen`) — P-1 Approved Mix + `numkeep.py`

- **Task (`AR-04`):** Generate the new chat SFT dataset from the frozen chat templates (`student_chat_template.txt` & `student_chat_template_hi.txt`) over non-excluded `train`/`dev` RHP passages (`[n] (RHP page p)`), filter all teacher outputs with `ml/handoff/eval/numkeep.py`, and mix with the existing risk-rewrite SFT data (`student_v1_train.jsonl.gz` 10,362 rows / `student_v1_dev.jsonl.gz` 529 rows).
- **Approved P-1 Chat SFT Mix (~3,000 chat rows total):**
  1. **1,500 scripted same-section hard near-miss unanswerables** (`label_source: "rule"`, EN 80% / HI 20%): multi-number passages from the right category where the requested metric/period/entity is verified absent $\to$ exact abstain string (`"I could not find this in the document."` / `"मुझे यह जानकारी दस्तावेज़ में नहीं मिली।"`).
  2. **~500 teacher-generated adversarial near-miss unanswerables** (`label_source: "ai"`): teacher crafts a plausible same-section question over `[n] (RHP page p)` blocks whose specific figure/period is not in the passages $\to$ exact abstain string.
  3. **~1,000 teacher-generated answerable numeric Q&A pairs** (`label_source: "ai"`): teacher writes a question and a 1–3 sentence answer copying numbers & units verbatim with `[n]` citations, filtered by `ml/handoff/eval/numkeep.py` (`finsight_lite.numcheck.unmatched_numbers`).
- **Expected Compute Units:**
  - **`DEBUG = True` (T4 smoke run):** ~4 min (~0.10 Colab units; generates 16 scripted + 12 teacher rows using `Qwen/Qwen3-4B-Instruct-2507` on T4, runs `numkeep.py`, and verifies the mixed SFT output).
  - **`DEBUG = False` (G4 full teacher run):** ~90 min (**13.07 G4 units** at 8.71 units/hr; checkpoints every chunk to `/content/drive/MyDrive/finsight_p4/m1_teacher_gen/epoch-<k>`).

In [ ]:
# First cell of every ARGON notebook (from ml/handoff/colab/setup_cell.py)
import json
import os
import pathlib
import random
import subprocess
import sys

HF_REPO = "AkshatTm/finsight-gpu-jobs"  # private dataset repo (C-ADR-13)
HF_PREFIX = "phase4/handoff"  # folder in that repo; revision pinned in ml/handoff/README.md
HF_REVISION = "a285902d91a5557cf6d6d7cf23a9e266410d6c3d"
JOB = "m1_teacher_gen"
DEBUG = True  # True = tiny slice for T4 debug run; set False on G4 ONLY after DEBUG passes
SEEDS = [13, 42, 2026]
RESUME = True  # continue from the newest checkpoint on Drive if one exists

ROOT = pathlib.Path("/content")
PACK = ROOT / "handoff"
DRIVE = pathlib.Path("/content/drive/MyDrive/finsight_p4")
CKPT_DIR = DRIVE / JOB


def mount_drive() -> None:
    try:
        from google.colab import drive  # type: ignore

        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive not mounted:", exc)
    CKPT_DIR.mkdir(parents=True, exist_ok=True)


def hf_token() -> str:
    try:
        from google.colab import userdata  # type: ignore

        return userdata.get("HF_TOKEN")
    except Exception:
        return os.environ["HF_TOKEN"]


def fetch_pack() -> pathlib.Path:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub"], check=True)
    from huggingface_hub import snapshot_download

    snapshot_download(
        repo_id=HF_REPO,
        repo_type="dataset",
        revision=HF_REVISION,
        token=hf_token(),
        allow_patterns=[f"{HF_PREFIX}/*"],
        local_dir=str(ROOT / "_hf"),
    )
    src = ROOT / "_hf" / HF_PREFIX
    if PACK.exists():
        subprocess.run(["rm", "-rf", str(PACK)], check=True)
    subprocess.run(["cp", "-r", str(src), str(PACK)], check=True)
    return PACK


def verify_manifest() -> None:
    import hashlib

    manifest = json.loads((PACK / "data" / "MANIFEST.json").read_text(encoding="utf-8"))
    for f in manifest["files"]:
        h = hashlib.sha256((PACK / "data" / f["path"]).read_bytes()).hexdigest()
        assert h == f["sha256"], f"checksum mismatch: {f['path']}"


def install_deps() -> None:
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "-r",
            str(PACK / "colab" / "requirements_colab.txt"),
        ],
        check=True,
    )


def set_seed(seed: int) -> None:
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import numpy as np

        np.random.seed(seed)
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    except ImportError:
        pass


def latest_checkpoint() -> pathlib.Path | None:
    ckpts = sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1]))
    return ckpts[-1] if RESUME and ckpts else None


mount_drive()
fetch_pack()
verify_manifest()
install_deps()
sys.path.insert(0, str(PACK / "eval" / "vendor"))
print("pack ready:", PACK, "| DEBUG =", DEBUG, "| resume from:", latest_checkpoint())

In [ ]:
# Cell 2: Load non-excluded train/dev risk passages and generate 1,500 scripted same-section near-miss unanswerables
import gzip
import time
from collections import defaultdict

t0_wall = time.time()
set_seed(SEEDS[0])

# Shallow-clone p4/argon if ml/m1_student_v2 is not in current working tree
repo_dir = pathlib.Path(".").resolve()
if not (repo_dir / "ml" / "m1_student_v2" / "chat_sft_builder.py").exists():
    repo_dir = ROOT / "repo"
    if not repo_dir.exists():
        subprocess.run(
            [
                "git",
                "clone",
                "--depth",
                "1",
                "-b",
                "p4/argon",
                "https://github.com/AkshatTm/FinSight-IPO-X-Ray-for-Indian-retail-investors.git",
                str(repo_dir),
            ],
            check=True,
        )
if str(repo_dir) not in sys.path:
    sys.path.insert(0, str(repo_dir))

from ml.m1_student_v2.chat_sft_builder import (  # noqa: E402
    ABSTAIN_EN,
    ABSTAIN_HI,
    build_scripted_near_miss_unanswerables,
    filter_teacher_chat_rows,
    format_passage_blocks,
    mix_rewrite_and_chat_sft,
    render_chat_prompt,
)

excluded_obj = json.loads((PACK / "data" / "excluded_ipos.json").read_text(encoding="utf-8"))
excluded_ipos = set(excluded_obj["ipos"].keys())

splits_obj = json.loads((PACK / "data" / "splits_by_ipo.json").read_text(encoding="utf-8"))
ipo_to_split = {
    item["ipo_id"]: item["slice"]
    for item in splits_obj["ipos"]
    if not item.get("excluded") and item["ipo_id"] not in excluded_ipos and item["slice"] in ("train", "dev")
}


def load_gz_jsonl(path: pathlib.Path) -> list[dict]:
    with gzip.open(path, "rt", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


risk_rows: list[dict] = []
for fname, default_split in [("classifier_train.jsonl.gz", "train"), ("classifier_dev.jsonl.gz", "dev")]:
    for r in load_gz_jsonl(PACK / "data" / fname):
        if r["ipo_id"] not in excluded_ipos and r["ipo_id"] in ipo_to_split:
            r["split"] = ipo_to_split[r["ipo_id"]]
            risk_rows.append(r)

for r in risk_rows:
    assert r["ipo_id"] not in excluded_ipos, f"Excluded IPO leaked: {r['ipo_id']}"

target_scripted = 16 if DEBUG else 1500
scripted_rows = build_scripted_near_miss_unanswerables(
    risk_rows=risk_rows,
    excluded_ipos=excluded_ipos,
    target_count=target_scripted,
    seed=SEEDS[0],
    handoff_dir=PACK,
)

scripted_path = CKPT_DIR / "scripted_near_miss_unanswerables.jsonl"
with scripted_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in scripted_rows:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

print(f"Built {len(scripted_rows)} scripted near-miss unanswerables -> {scripted_path}")

In [ ]:
# Cell 3: Teacher generation (~1,150 raw answerable numeric Q&A + ~500 adversarial near-miss unanswerables)
# Checkpoints every chunk (`epoch-<k>`) to CKPT_DIR so RESUME=True continues seamlessly.
import re
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# In DEBUG mode on T4, use the fast 4B model for smoke testing; on G4 (DEBUG=False), use the teacher model.
TEACHER_MODEL = "Qwen/Qwen3-4B-Instruct-2507" if DEBUG else os.environ.get("FINSIGHT_TEACHER_MODEL", "google/gemma-3-27b-it")
TARGET_ANSWERABLE_RAW = 8 if DEBUG else 1150  # ~1,150 raw -> >=1,000 kept after numkeep.py
TARGET_ADV_UNANS = 4 if DEBUG else 500
CHUNK_SIZE = 12 if DEBUG else 50

# Build passage bundles (2-3 multi-number passages from the same non-excluded IPO & category)
num_re = re.compile(r"(?:₹|rs\.?|inr|\d+(?:\.\d+)?\s*(?:%|million|crore|lakh))", re.I)
by_ipo_cat: dict[tuple[str, str], list[dict]] = defaultdict(list)
for r in risk_rows:
    txt = " ".join(str(r.get("text") or "").split())
    if len(txt.split()) >= 30 and num_re.search(txt):
        cat = str(r.get("label") or r.get("category") or "financial")
        by_ipo_cat[(r["ipo_id"], cat)].append(
            {
                "risk_id": r.get("risk_id") or r["ipo_id"],
                "ipo_id": r["ipo_id"],
                "company": r.get("company") or r["ipo_id"],
                "split": r["split"],
                "page": int(r.get("page") or 45),
                "doc_type": "RHP",
                "text": txt,
            }
        )

rng = random.Random(SEEDS[1])
passage_bundles: list[dict] = []
for (ipo_id, cat), plist in sorted(by_ipo_cat.items()):
    if len(plist) < 2:
        continue
    rng.shuffle(plist)
    for i in range(0, len(plist) - 1, 2):
        chunk_p = plist[i : i + 2]
        passage_bundles.append(
            {
                "ipo_id": ipo_id,
                "company": chunk_p[0]["company"],
                "split": chunk_p[0]["split"],
                "category": cat,
                "passage_objs": chunk_p,
            }
        )
rng.shuffle(passage_bundles)

total_needed = TARGET_ANSWERABLE_RAW + TARGET_ADV_UNANS
selected_bundles = (passage_bundles * ((total_needed // max(1, len(passage_bundles))) + 1))[:total_needed]

# Resume from existing epoch-<k> checkpoints if present
generated_raw: list[dict] = []
completed_chunks = 0
if RESUME:
    for ckpt in sorted(CKPT_DIR.glob("epoch-*"), key=lambda p: int(p.name.split("-")[1])):
        part_file = ckpt / "teacher_raw_chunk.jsonl"
        if part_file.exists():
            with part_file.open("r", encoding="utf-8") as f:
                generated_raw.extend(json.loads(line) for line in f if line.strip())
            completed_chunks = int(ckpt.name.split("-")[1])
    if generated_raw:
        print(f"Resumed {len(generated_raw)} teacher rows from {completed_chunks} completed chunks.")

if len(generated_raw) < total_needed:
    print(f"Loading teacher model: {TEACHER_MODEL}")
    tok = AutoTokenizer.from_pretrained(TEACHER_MODEL, token=hf_token())
    load_kwargs = {"device_map": "auto", "token": hf_token(), "torch_dtype": torch.bfloat16}
    if not DEBUG:
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", "bitsandbytes"], check=True)
            from transformers import BitsAndBytesConfig

            load_kwargs["quantization_config"] = BitsAndBytesConfig(
                load_in_4bit=True,
                bnb_4bit_compute_dtype=torch.bfloat16,
                bnb_4bit_quant_type="nf4",
            )
        except Exception as exc:
            print("4-bit quantization fallback to bfloat16:", exc)
    teacher_model = AutoModelForCausalLM.from_pretrained(TEACHER_MODEL, **load_kwargs)
    teacher_model.eval()

    def generate_teacher_json(system_Instruction: str, blocks_text: str) -> dict | None:
        messages = [
            {"role": "user", "content": f"{system_Instruction}\n\nPassages:\n<<<DATA\n{blocks_text}\nDATA>>>\n\nJSON:"}
        ]
        prompt_ids = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(teacher_model.device)
        with torch.no_grad():
            out_ids = teacher_model.generate(
                prompt_ids,
                max_new_tokens=220,
                do_sample=False,
                pad_token_id=tok.eos_token_id,
            )
        gen_text = tok.decode(out_ids[0][prompt_ids.shape[1] :], skip_special_tokens=True).strip()
        m = re.search(r"\{.*\}", gen_text, re.S)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    ANS_INSTRUCTION = (
        "You are generating grounded Q&A training data over Indian IPO RHP passages. "
        "Write ONE specific financial or operational question answered by the passages, and a 1-2 sentence answer "
        "that copies every number AND its unit verbatim (e.g. '₹ 4,720.00 million' or '14.25%') and cites the passage "
        "like [1] or [2] at the end of every sentence. Return ONLY valid JSON: {\"question\": \"...\", \"answer\": \"...\"}."
    )
    UNANS_INSTRUCTION = (
        "You are generating hard near-miss unanswerable questions over Indian IPO RHP passages. "
        "Write ONE plausible investor question on the same topic as the passages, but asking for a specific metric, "
        "fiscal year, or counterparty that is NOT mentioned anywhere in the passages. "
        "Return ONLY valid JSON: {\"question\": \"...\"}."
    )

    start_idx = len(generated_raw)
    chunk_num = completed_chunks
    for batch_start in range(start_idx, total_needed, CHUNK_SIZE):
        chunk_num += 1
        chunk_rows: list[dict] = []
        for idx in range(batch_start, min(batch_start + CHUNK_SIZE, total_needed)):
            b = selected_bundles[idx]
            blocks_str = format_passage_blocks(b["passage_objs"])
            is_ans = idx < TARGET_ANSWERABLE_RAW
            lang = "hi" if (idx % 5 == 0) else "en"
            if is_ans:
                parsed = generate_teacher_json(ANS_INSTRUCTION, blocks_str)
                if parsed and parsed.get("question") and parsed.get("answer"):
                    chunk_rows.append(
                        {
                            "id": f"chat_ans_teacher_{idx + 1:05d}",
                            "ipo_id": b["ipo_id"],
                            "company": b["company"],
                            "split": b["split"],
                            "task_type": "chat_answerable_teacher",
                            "lang": "en",
                            "question": str(parsed["question"]).strip(),
                            "answer": str(parsed["answer"]).strip(),
                            "passage_objs": b["passage_objs"],
                            "passages": [blocks_str for _ in ["text"] for _ in [1] if b["passage_objs"]],
                        }
                    )
            else:
                parsed = generate_teacher_json(UNANS_INSTRUCTION, blocks_str)
                if parsed and parsed.get("question"):
                    abstain_target = ABSTAIN_HI if lang == "hi" else ABSTAIN_EN
                    chunk_rows.append(
                        {
                            "id": f"chat_unans_teacher_{idx + 1:05d}",
                            "ipo_id": b["ipo_id"],
                            "company": b["company"],
                            "split": b["split"],
                            "task_type": "chat_unanswerable_teacher",
                            "lang": lang,
                            "question": str(parsed["question"]).strip(),
                            "answer": abstain_target,
                            "passage_objs": b["passage_objs"],
                            "passages": [blocks_str for _ in ["text"] for _ in [1] if b["passage_objs"]],
                        }
                    )

        ckpt_folder = CKPT_DIR / f"epoch-{chunk_num}"
        ckpt_folder.mkdir(parents=True, exist_ok=True)
        with (ckpt_folder / "teacher_raw_chunk.jsonl").open("w", encoding="utf-8", newline="\n") as f:
            for r in chunk_rows:
                f.write(json.dumps(r, ensure_ascii=False) + "\n")
        generated_raw.extend(chunk_rows)
        print(f"[Chunk epoch-{chunk_num}] Saved {len(chunk_rows)} rows (total={len(generated_raw)}/{total_needed})")

In [ ]:
# Cell 4: Run ml/handoff/eval/numkeep.py + citation/length filter, mix with student_v1_{train,dev}.jsonl.gz, save SFT sets
import datetime

raw_teacher_path = CKPT_DIR / "teacher_raw.jsonl"
with raw_teacher_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in generated_raw:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 1. Run CC's exact ml/handoff/eval/numkeep.py script on teacher_raw.jsonl
numkeep_out_path = CKPT_DIR / "teacher_numkeep_only.jsonl"
numkeep_proc = subprocess.run(
    [
        sys.executable,
        str(PACK / "eval" / "numkeep.py"),
        "--in",
        str(raw_teacher_path),
        "--out",
        str(numkeep_out_path),
        "--source-field",
        "passages",
        "--output-field",
        "answer",
    ],
    capture_output=True,
    text=True,
    check=True,
)
numkeep_cli_stats = json.loads(numkeep_proc.stdout.strip())
print("numkeep.py CLI stats:", numkeep_cli_stats)

# 2. Apply full filter_teacher_chat_rows (numkeep + [n] citation + <=120 words + frozen chat prompt rendering)
kept_teacher, rejected_teacher, full_filter_stats = filter_teacher_chat_rows(
    raw_rows=generated_raw,
    excluded_ipos=excluded_ipos,
    handoff_dir=PACK,
)

kept_path = CKPT_DIR / "teacher_chat_kept.jsonl"
rej_path = CKPT_DIR / "teacher_chat_kept.rejected.jsonl"
with kept_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in kept_teacher:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with rej_path.open("w", encoding="utf-8", newline="\n") as f:
    for r in rejected_teacher:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

# 3. Mix with existing rewrite pairs (student_v1_train.jsonl.gz 10,362 / student_v1_dev.jsonl.gz 529)
rw_train = load_gz_jsonl(PACK / "data" / "student_v1_train.jsonl.gz")
rw_dev = load_gz_jsonl(PACK / "data" / "student_v1_dev.jsonl.gz")
if DEBUG:
    rw_train = rw_train[:32]
    rw_dev = rw_dev[:16]

all_chat_rows = scripted_rows + kept_teacher
combined_train, combined_dev, mix_summary = mix_rewrite_and_chat_sft(
    rewrite_train=rw_train,
    rewrite_dev=rw_dev,
    chat_rows=all_chat_rows,
    excluded_ipos=excluded_ipos,
    seed=SEEDS[0],
)

train_out = CKPT_DIR / "student_v2_sft_train.jsonl.gz"
dev_out = CKPT_DIR / "student_v2_sft_dev.jsonl.gz"
with gzip.open(train_out, "wt", encoding="utf-8", newline="\n") as f:
    for r in combined_train:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
with gzip.open(dev_out, "wt", encoding="utf-8", newline="\n") as f:
    for r in combined_dev:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")

wall_minutes = round((time.time() - t0_wall) / 60.0, 2)
hw_name = "T4-DEBUG" if DEBUG else "G4"
units_rate = 1.60 if DEBUG else 8.71
units_used = round((wall_minutes / 60.0) * units_rate, 2)

gen_report = {
    "job": JOB,
    "debug": DEBUG,
    "teacher_model": TEACHER_MODEL,
    "numkeep_cli_stats": numkeep_cli_stats,
    "full_filter_stats": full_filter_stats,
    "mix_summary": mix_summary,
    "compute": {
        "hardware": hw_name,
        "units_used": units_used,
        "wall_minutes": wall_minutes,
    },
    "date": datetime.date.today().isoformat(),
}
(CKPT_DIR / "generation_stats.json").write_text(json.dumps(gen_report, indent=2), encoding="utf-8")
print(json.dumps(gen_report, indent=2))